# Resource validation of module `monaqa2.qiskit2`


## Import 

In [10]:
import sys
from itertools import product
import numpy as np
from qiskit import transpile
from qiskit.circuit import QuantumCircuit, Instruction

if ".." not in sys.path:
    sys.path.append("..")

from monaqa2.qiskit2 import (
    Ccx, Rx, Ry, Crx, Cry, Crz, Ccrx, Ccry, Ccrz, Rzz, Crzz, Ccrzz,
    GivensRotation, ControlledGivensRotation, ReversibleFanout, ReversibleZeroFlag,
    ProposalLocal, ProposalUniform, ProposalHamSim, ProposalHamSimRandomized,
    ReversibleCarryLookaheadAdder, ReversibleDeltaEnergy, ReversiblePositivePart,
    ReversibleComparator, RotationalAcceptance, BoltzmannCoin, AcceptPath, Reflection, Walk,
)

## Resource estimation helpers

The following methods are used to estimate the depth of the quantum circuit. We must avoid built in decomposition methods of qiskit as they might optimize for the size and not the depth of the circuit. That is why we explicitly give a known decomposition for the instruction that qiskit already gives as a primitive. 

The method used in the actual estimation is `get_nc_depth`, which decompose the circuit in Clifford+RZ on assumed all-to-all connectivity.

We first define the basis of operator we are working with. We need to include barrier and other auxiliary instructions, they can be removed later on. 

In [11]:
_CLIFFORD_RZ_BASIS = {
    "cx", "h", "s", "sdg", "t", "tdg", "x", "z", "rz",
    "barrier", "measure", "id",
}

Some $R_Z$ rotations are close to a Clifford gate, we can explicitly remove them. 

In [12]:
def _replace_clifford_rz(qc: QuantumCircuit) -> QuantumCircuit:
    
    out = QuantumCircuit(*qc.qregs, *qc.cregs, name=qc.name)
    out.global_phase = qc.global_phase

    for inst in qc.data:
        op, qargs, cargs = inst.operation, inst.qubits, inst.clbits

        if op.name == "rz" and len(op.params) == 1:
            try:
                theta = float(op.params[0])
            except TypeError:
                out.append(op, qargs, cargs)
                continue

            k = int(np.round(2 * theta / np.pi))

            if np.isclose(theta, k * np.pi / 2, atol=1e-10, rtol=0):
                out.global_phase += -k * np.pi / 4

                if k % 4 == 1:
                    out.s(qargs[0])
                elif k % 4 == 2:
                    out.z(qargs[0])
                elif k % 4 == 3:
                    out.sdg(qargs[0])

                continue

        out.append(op, qargs, cargs)

    return out

The decomposition is done explicitly, avoid using transpile:

In [13]:
def _append_recursively_decomposed(
    out: QuantumCircuit,
    op: Instruction,
    qargs,
    cargs,
    basis: set[str],
) -> None:
    if op.name in basis:
        out.append(op, qargs, cargs)
        return

    if op.definition is not None:
        definition = op.definition

        # A global phase in the definition is part of the decomposition.
        out.global_phase += definition.global_phase

        for subinst in definition.data:
            subop = subinst.operation
            subqargs = [qargs[definition.find_bit(q).index] for q in subinst.qubits]
            subcargs = [cargs[definition.find_bit(c).index] for c in subinst.clbits]

            _append_recursively_decomposed(
                out=out,
                op=subop,
                qargs=subqargs,
                cargs=subcargs,
                basis=basis,
            )

        return

    # Last-resort fallback: synthesize only this one opaque operation,
    # not the whole circuit.
    tmp = QuantumCircuit(len(qargs), len(cargs))
    tmp.append(op, range(len(qargs)), range(len(cargs)))

    tmp_u = transpile(
        tmp,
        basis_gates=sorted(basis - {"barrier", "measure", "id"}),
        optimization_level=0,
        seed_transpiler=1234,
        output_name="",
    )

    out.global_phase += tmp_u.global_phase

    for subinst in tmp_u.data:
        subop = subinst.operation
        subqargs = [qargs[tmp_u.find_bit(q).index] for q in subinst.qubits]
        subcargs = [cargs[tmp_u.find_bit(c).index] for c in subinst.clbits]

        _append_recursively_decomposed(
            out=out,
            op=subop,
            qargs=subqargs,
            cargs=subcargs,
            basis=basis,
        )


def recursively_decompose_to_clifford_rz(
    qc: QuantumCircuit,
    basis: set[str] | None = None,
) -> QuantumCircuit:
    basis = _CLIFFORD_RZ_BASIS if basis is None else basis

    out = QuantumCircuit(*qc.qregs, *qc.cregs, name=qc.name)
    out.global_phase = qc.global_phase

    for inst in qc.data:
        _append_recursively_decomposed(
            out=out,
            op=inst.operation,
            qargs=inst.qubits,
            cargs=inst.clbits,
            basis=basis,
        )

    return out

The composition of the two methods results in our own transpilation method:

In [14]:
def qiskit_to_clifford_rz(qc: QuantumCircuit, seed: int = 1234, opt: int = 0) -> QuantumCircuit:
    qc_u = recursively_decompose_to_clifford_rz(qc)
    return _replace_clifford_rz(qc_u)

Then the depth is obtained by getting the depth of the circuit once the clifford gates are eliminated. 

In [15]:
def get_nc_depth(qc: QuantumCircuit, transpile: bool = True) -> int:
    if transpile:
        qc = qiskit_to_clifford_rz(qc)
    filter = lambda inst: inst.operation.name in ["t", "tdg", "rz"]
    return int(qc.depth(filter_function=filter))

The method `validate_resource` checks measured depth against its upper bound.

In [16]:
def validate_resource(operation, upper_bound_depth, verbose=False):
    qc = operation if isinstance(operation, QuantumCircuit) else operation.definition
    depth = get_nc_depth(qc)
    assert depth <= upper_bound_depth, (operation.name, depth, upper_bound_depth)
    if verbose:
        print(f"{operation.name}: {depth} <= {upper_bound_depth}")
    return depth

For our tests, one instance per $n$ is randomly generated. That is not a limitation, since the instance just influences the precision parameter via the 1-norm. 

In [17]:
def instance(n):
    rng = np.random.default_rng(n)
    h = rng.normal(size=n) / np.sqrt(n)
    J = np.triu(rng.normal(size=(n, n)), 1) / np.sqrt(n)
    return h, J + J.T

The $\ell_x$ notation is a quicker way to denote the base-two logarithm of $x$, it is used in many formulas below. 

In [18]:
def ell(x):
    return (int(x) - 1).bit_length()

We anticipate the depth upper bounds used below:

In [19]:
def wallace_depth(M, W):
    return 18 * int(np.ceil(np.log(M) / np.log(1.5))) + 12 * ell(W) + 6


def delta_depth(n, W):
    M = n * (n + 1) // 2
    return 18 * int(np.ceil(np.log(M) / np.log(1.5))) + 12 * ell(W) + 18


def coin_depth(n, W):
    M = n * (n + 1) // 2
    return 36 * int(np.ceil(np.log(M) / np.log(1.5))) + 24 * ell(W) + 44


def ham_depth(n, r, k=0):
    return (1 + 4 * k + 8 * k**2) * (r * (n + 2) + 1)

## Primitives

$m$ is the number of input qubits tested by `ReversibleZeroFlag`, excluding its flag and work qubits. For fanout, $m$ counts all wires: one source and $m-1$ copies.

| Element | simple upper bound depth | asymp |
|---|---:|---|
| Ccx | $3$ | $O(1)$ |
| Rx, Ry, Rzz | $1$ | $O(1)$ |
| Crx, Cry, Crz, Crzz | $2$ | $O(1)$ |
| Ccrx, Ccry, Ccrz, Ccrzz | $8$ | $O(1)$ |
| GivensRotation | $2$ | $O(1)$ |
| ControlledGivensRotation | $14$ | $O(1)$ |
| ReversibleFanout | $0$ | $0$ |
| ReversibleZeroFlag | $6\lceil\log_2 m\rceil$ | $O(\log m)$ |


In [20]:
for gate, bound in [
    (Ccx(), 3), (Rx(.1), 1), (Ry(.1), 1), (Rzz(.1), 1),
    (Crx(.1), 2), (Cry(.1), 2), (Crz(.1), 2), (Crzz(.1), 2),
    (Ccrx(.1), 8), (Ccry(.1), 8), (Ccrz(.1), 8), (Ccrzz(.1), 8),
    (GivensRotation(.3, .7), 2), (ControlledGivensRotation(.3, .7), 14),
]:
    validate_resource(gate, bound, verbose=True)
for m in (1, 2, 3, 4, 8, 16, 32, 64, 128, 256, 512, 1024):
    validate_resource(ReversibleFanout(m), 0)
    validate_resource(ReversibleZeroFlag(m), 6 * ell(m))
print("Control-tree checks passed.")


ccx: 3 <= 3
rx: 1 <= 1
ry: 1 <= 1
rzz: 1 <= 1
crx: 2 <= 2
cry: 2 <= 2
crz: 2 <= 2
crzz: 2 <= 2
ccrx: 8 <= 8
ccry: 8 <= 8
ccrz: 8 <= 8
ccrzz: 8 <= 8
givens: 2 <= 2
c_givens: 14 <= 14
Control-tree checks passed.


## Proposals

$n$ is the number of spins; $r$ is the number of Trotter steps. For randomized proposals, $k\ge1$ is the number of bits per time ($t$) and mixing-weight ($\gamma$) register, giving $2^k$ grid values for each parameter.

| Element | simple upper bound depth | asymp |
|---|---:|---|
| ProposalUniform | $0$ | $0$ |
| ProposalLocal | $2\lceil\log_2 n\rceil$ | $O(\log n)$ |
| ProposalHamSim | $r(n+2)+1$ | $O(rn)$ |
| ProposalHamSimRandomized | $(1+4k+8k^2)[r(n+2)+1]$ | $O(rnk^2)$ |


In [21]:
for n in (1, 2, 3, 4, 8, 16, 32, 64, 128, 256, 512):
    validate_resource(ProposalUniform(n), 0)
    validate_resource(ProposalLocal(n), 2 * ell(n), verbose=True)
    
for n, r in product((2, 3, 4, 8, 16, 32, 64), (1, 2, 4)):
    h, J = instance(n)
    validate_resource(ProposalHamSim(n, h, J, .713, .413, r), ham_depth(n, r), verbose=True)
    
for n, k, r in product((2, 4, 8, 16), (1, 2, 3), (1, 2)):
    h, J = instance(n)
    gate = ProposalHamSimRandomized(n, h, J, (.37, 1.13), (.17, .71), r, k)
    print(f"n={n}, k={k}, r={r}: ", end="")
    validate_resource(gate, ham_depth(n, r, k), verbose=True)

ProposalLocal(n=1): 0 <= 0
ProposalLocal(n=2): 2 <= 2
ProposalLocal(n=3): 4 <= 4
ProposalLocal(n=4): 4 <= 4
ProposalLocal(n=8): 6 <= 6
ProposalLocal(n=16): 8 <= 8
ProposalLocal(n=32): 10 <= 10
ProposalLocal(n=64): 12 <= 12
ProposalLocal(n=128): 14 <= 14
ProposalLocal(n=256): 16 <= 16
ProposalLocal(n=512): 18 <= 18
ProposalHamSim(n=2): 4 <= 5
ProposalHamSim(n=2): 7 <= 9
ProposalHamSim(n=2): 13 <= 17
ProposalHamSim(n=3): 6 <= 6
ProposalHamSim(n=3): 11 <= 11
ProposalHamSim(n=3): 21 <= 21
ProposalHamSim(n=4): 6 <= 7
ProposalHamSim(n=4): 11 <= 13
ProposalHamSim(n=4): 21 <= 25
ProposalHamSim(n=8): 10 <= 11
ProposalHamSim(n=8): 19 <= 21
ProposalHamSim(n=8): 37 <= 41
ProposalHamSim(n=16): 18 <= 19
ProposalHamSim(n=16): 35 <= 37
ProposalHamSim(n=16): 69 <= 73
ProposalHamSim(n=32): 34 <= 35
ProposalHamSim(n=32): 67 <= 69
ProposalHamSim(n=32): 133 <= 137
ProposalHamSim(n=64): 66 <= 67
ProposalHamSim(n=64): 131 <= 133
ProposalHamSim(n=64): 261 <= 265
n=2, k=1, r=1: ProposalHamSimRandomized(n=2): 5

## Boltzmann coin

$M=n(n+1)/2$ counts the $n$ field terms and $n(n-1)/2$ pair terms, including zero coefficients; for the Wallace adder it is the number of input words. $W$ is the actual number of qubits in the signed energy word, read from `delta.W`. The bounds cover both zero and positive beta and at most $\lceil\log_{3/2}M\rceil$ Wallace rounds.

| Element | simple upper bound depth | asymp |
|---|---:|---|
| Conditional loader | $6$ | $O(1)$ |
| Carry-lookahead adder | $6+12\lceil\log_2 W\rceil$ | $O(\log W)$ |
| Wallace adder | $18\lceil\log_{3/2}M\rceil+12\lceil\log_2 W\rceil+6$ | $O(\log M+\log W)$ |
| Delta energy | $18\left\lceil\log_{3/2}\frac{n(n+1)}2\right\rceil+12\lceil\log_2 W\rceil+18$ | $O(\log n+\log W)$ |
| Positive part | $3$ | $O(1)$ |
| Comparator | $0$ | $0$ |
| Rotational acceptance | $2$ | $O(1)$ |
| BoltzmannCoin | $36\left\lceil\log_{3/2}\frac{n(n+1)}2\right\rceil+24\lceil\log_2 W\rceil+44$ | $O(\log n+\log W)$ |


In [22]:
for W in (1, 2, 3, 4, 8, 16, 32, 64, 128, 256):
    validate_resource(ReversibleCarryLookaheadAdder(W), 6 + 12 * ell(W))
    
for n, F in product((1, 2, 3, 4, 8, 16), (4, 8, 12)):
    h, J = instance(n)
    delta = ReversibleDeltaEnergy(n, h, J, F)
    W, M = delta.W, n * (n + 1) // 2
    
    for gate, bound in [
        (delta.loader, 6), (delta.adder, wallace_depth(M, W)),
        (delta, delta_depth(n, W)), (ReversiblePositivePart(W), 3),
        (ReversibleComparator(W), 0),
    ]:
        validate_resource(gate, bound)
    for beta in (0., 4.):
        validate_resource(RotationalAcceptance(W, beta, delta.energy_unit), 2)
        print(f"n={n}, F={F}, W={W}, beta={beta}: ", end="")
        validate_resource(BoltzmannCoin(n, h, J, F, beta), coin_depth(n, W), verbose=True)


n=1, F=4, W=5, beta=0.0: boltzmann_coin: 24 <= 116
n=1, F=4, W=5, beta=4.0: boltzmann_coin: 25 <= 116
n=1, F=8, W=9, beta=0.0: boltzmann_coin: 24 <= 140
n=1, F=8, W=9, beta=4.0: boltzmann_coin: 25 <= 140
n=1, F=12, W=13, beta=0.0: boltzmann_coin: 24 <= 140
n=1, F=12, W=13, beta=4.0: boltzmann_coin: 25 <= 140
n=2, F=4, W=4, beta=0.0: boltzmann_coin: 100 <= 200
n=2, F=4, W=4, beta=4.0: boltzmann_coin: 100 <= 200
n=2, F=8, W=9, beta=0.0: boltzmann_coin: 132 <= 248
n=2, F=8, W=9, beta=4.0: boltzmann_coin: 133 <= 248
n=2, F=12, W=13, beta=0.0: boltzmann_coin: 132 <= 248
n=2, F=12, W=13, beta=4.0: boltzmann_coin: 133 <= 248
n=3, F=4, W=5, beta=0.0: boltzmann_coin: 188 <= 296
n=3, F=4, W=5, beta=4.0: boltzmann_coin: 189 <= 296
n=3, F=8, W=9, beta=0.0: boltzmann_coin: 204 <= 320
n=3, F=8, W=9, beta=4.0: boltzmann_coin: 205 <= 320
n=3, F=12, W=13, beta=0.0: boltzmann_coin: 204 <= 320
n=3, F=12, W=13, beta=4.0: boltzmann_coin: 205 <= 320
n=4, F=4, W=5, beta=0.0: boltzmann_coin: 224 <= 332
n=4, F

## Reflection and Accept Path

$C$ is the number of persistent coin qubits ($C=W-1$ for `BoltzmannCoin`). $E$ is the number of extra proposal wires included in the reflection: parameter registers and their fanout copies. Here $E=0$ for local, uniform, and fixed Hamiltonian proposals, and $E=2kn$ for randomized proposals.

| Element | simple upper bound depth | asymp |
|---|---:|---|
| AcceptPath | $12\lceil\log_2 C\rceil+3$ | $O(\log C)$ |
| Reflection | $12\lceil\log_2(n+C+E)\rceil-6$ | $O(\log(n+C+E))$ |


In [23]:
for n, C in product((1, 2, 3, 4, 8, 16, 32, 64, 128), (1, 2, 4, 8, 16, 32, 64)):
    validate_resource(AcceptPath(n, C), 12 * ell(C) + 3)
    for E in (0, 4 * n):
        validate_resource(Reflection(n, C, E), 12 * ell(n + C + E) - 6)
        
print("Reflection and accept-path checks passed for all n, C, E.")

Reflection and accept-path checks passed for all n, C, E.


## Complete walk

`eps`=$\varepsilon_b>0$ is the tolerance used to select the coefficient precision,

$$F=\max(0,\lceil\log_2(\beta\|H\|_1n(n+1)/(4\varepsilon_b))\rceil).$$

Here $\|H\|_1$ is the Ising Pauli-coefficient 1-norm; take $F=0$ when $\beta\|H\|_1=0$.

The tables use the resulting signed word width $W=\texttt{walk.coin.W}$, with $n,r,k$ as defined above.

| Proposal | simple upper bound depth | asymp |
|---|---:|---|
| Local / uniform | $72\left\lceil\log_{3/2}\frac{n(n+1)}2\right\rceil+76\lceil\log_2(n+W)\rceil+85$ | $O(\log n+\log W)$ |
| Hamiltonian | $2r(n+2)+72\left\lceil\log_{3/2}\frac{n(n+1)}2\right\rceil+72\lceil\log_2(n+W)\rceil+87$ | $O(rn+\log W)$ |
| Randomized Hamiltonian | $2(1+4k+8k^2)[r(n+2)+1]+72\left\lceil\log_{3/2}\frac{n(n+1)}2\right\rceil+72\lceil\log_2(n(2k+1)+W)\rceil+85$ | $O(rnk^2+\log W)$ |


In [6]:
eps, beta, k, r = .1, 4., 2, 2

for n, proposal in product((1, 2, 3, 4, 8, 16), ("local", "uniform", "hamsim", "hamsim_randomized")):
    
    h, J = instance(n)
    
    walk = Walk(h, J, eps, beta, proposal, k, t=.713, gamma=.413, trotter_steps=r)
    F, W = walk.F, walk.coin.W
    norm_1 = np.abs(h).sum() + np.abs(np.triu(J, 1)).sum()
    
    assert F == max(0, int(np.ceil(np.log2(beta * norm_1 * n * (n + 1) / (4 * eps)))))
    
    M = n * (n + 1) // 2
    bound = 72 * int(np.ceil(np.log(M) / np.log(1.5))) + 85
    if proposal in ("local", "uniform"):
        bound += 76 * ell(n + W)
    elif proposal == "hamsim":
        bound += 2 * r * (n + 2) + 72 * ell(n + W) + 2
    else:
        bound += 2 * (1 + 4 * k + 8 * k**2) * (r * (n + 2) + 1) + 72 * ell(n * (2 * k + 1) + W)
    
    print(f"n={n}, proposal={proposal}, F={F}, W={W}: ", end="")
    validate_resource(walk, bound, verbose=True)

n=1, proposal=local, F=3, W=4: walk: 71 <= 313
n=1, proposal=uniform, F=3, W=4: walk: 71 <= 313
n=1, proposal=hamsim, F=3, W=4: walk: 81 <= 315
n=1, proposal=hamsim_randomized, F=3, W=4: walk: 493 <= 947
n=2, proposal=local, F=8, W=9: walk: 313 <= 605
n=2, proposal=uniform, F=8, W=9: walk: 309 <= 605
n=2, proposal=hamsim, F=8, W=9: walk: 323 <= 607
n=2, proposal=hamsim_randomized, F=8, W=9: walk: 895 <= 1399
n=3, proposal=local, F=9, W=10: walk: 461 <= 749
n=3, proposal=uniform, F=9, W=10: walk: 453 <= 749
n=3, proposal=hamsim, F=9, W=10: walk: 475 <= 755
n=3, proposal=hamsim_randomized, F=9, W=10: walk: 1367 <= 1707
n=4, proposal=local, F=10, W=11: walk: 605 <= 821
n=4, proposal=uniform, F=10, W=11: walk: 597 <= 821
n=4, proposal=hamsim, F=10, W=11: walk: 619 <= 831
n=4, proposal=hamsim_randomized, F=10, W=11: walk: 1511 <= 1943
n=8, proposal=local, F=14, W=14: walk: 837 <= 1113
n=8, proposal=uniform, F=14, W=14: walk: 825 <= 1113
n=8, proposal=hamsim, F=14, W=14: walk: 863 <= 1135
n=